# In-class Activity 14: Sentiment Analysis on Real Reviews

## Part 1: Load and Explore the Data
### 2. Load the data

In [1]:
import pandas as pd

DATA_PATH = "data/amazon_reviews_us_Digital_Video_Games_v1_00.tsv.zip"

df = pd.read_csv(DATA_PATH, sep="\t")
print("Default settings:", df.shape)

Default settings: (144724, 15)


**How many rows and columns did pandas load?**

With the default settings pandas loaded 144,724 rows and 15 columns.

### 3. Load it again correctly

In [2]:
df = pd.read_csv(DATA_PATH, sep="\t", quoting=3)  # 3 = treat quotes as plain text
print("quoting=3:", df.shape)

quoting=3: (145431, 15)


**How many rows does each version load? Did pandas warn you about the difference?**

The default version loaded 144,724 rows and the quoting=3 version loaded 145,431 rows, so 707 reviews were missing the first time. Pandas did not give any warning.

**What does that tell you about checking your data?**

It shows that you can't just trust that the data loaded correctly because there was no error. You should check things like the row count and actually look at the data before analyzing it, because a silent problem like this would throw off every result after it.

### 4. Read some actual reviews

In [3]:
peek = df["review_body"].sample(5, random_state=2134)
for text in peek:
    print(text, "\n")

The &#34;AI&#34; is wrong by cheating, many bugs and control system cheating is not reliable<br />the PC version is penalized 

Is fast secure and very easy you only payed and in less than a minute you can use the code thanks 

This is exactly what it says it is, and I was able to email it as a gift!  Perfect.  Thanks! 

Since there is no way to return downloaded games or even remark on my problem, my only other options is via a review.<br /><br />I have a steam account and am logged in.  I tried installing the game, and it gave me a generic &#34;Trouble Communicating With Gateway&#34; message.  Googling the message returns one web page in German.  Retried three times with the same result.<br /><br />So...  Looks like I'm screwed. 

I bought this crapware online but was not able to install it on my tablet nor my android phone. The amazon android market app \\"seems\\" to crash when installing free or promotional apps. Amazon customer service is non existent. They dont care once they ha

**What do you notice that looks weird in these reviews?**

Some of the reviews have things like `&#34;` instead of quote marks, `<br />` in the middle of sentences, and quotes that show up as `\\"`.

**Where do you think it came from?**

It most likely came from the website itself, since `<br />` is an HTML line break and `&#34;` is the HTML code for a quote mark. When the reviews were exported from Amazon's site, the HTML formatting got saved along with the text.

### 5. Look at the star ratings

In [4]:
df["star_rating"].value_counts(normalize=True).sort_index()

star_rating
1    0.171697
2    0.053283
3    0.079962
4    0.140314
5    0.554744
Name: proportion, dtype: float64

**How are the ratings distributed? Why might that matter when you measure a model's accuracy?**

The ratings are very uneven. About 55% of reviews are 5 stars and 17% are 1 star, while 2, 3 and 4 stars only make up about 5%, 8% and 14%. This matters because a model could get a high accuracy just by guessing "positive" most of the time, so the accuracy number would look better than the model actually is.

### 6. Look at the products

In [5]:
df["product_title"].value_counts().head(10)

product_title
Playstation Network Card                                          13642
Xbox Live Subscription                                             7307
Playstation Plus Subscription                                      4712
Xbox Live Gift Card                                                3438
SimCity - Limited Edition                                          3421
Xbox 360 Live Points Card                                          3036
Battlefield 4                                                      1453
Final Fantasy XIV: A Realm Reborn                                  1273
Battlefield 3                                                      1160
Command and Conquer The Ultimate Collection [Online Game Code]      875
Name: count, dtype: int64

**What do you notice about the most-reviewed "games"?**

Most of them aren't actually games. The top four are a Playstation Network Card, Xbox Live Subscription, Playstation Plus Subscription and Xbox Live Gift Card, so they are gift cards and subscriptions. Only a few on the list, like SimCity, Battlefield 4 and Final Fantasy XIV, are real games.

### 7. Clean the text

In [6]:
print("Reviews with <br />:", df["review_body"].str.contains("<br />").sum())
df = df.dropna(subset=["review_body"])
df["review_body"] = df["review_body"].str.replace("<br />", " ")
print("After cleaning:", df["review_body"].str.contains("<br />").sum(), "\n")

for text in df.loc[peek.index, "review_body"]:
    print(text, "\n")

Reviews with <br />: 25699
After cleaning: 0 

The &#34;AI&#34; is wrong by cheating, many bugs and control system cheating is not reliable the PC version is penalized 

Is fast secure and very easy you only payed and in less than a minute you can use the code thanks 

This is exactly what it says it is, and I was able to email it as a gift!  Perfect.  Thanks! 

Since there is no way to return downloaded games or even remark on my problem, my only other options is via a review.  I have a steam account and am logged in.  I tried installing the game, and it gave me a generic &#34;Trouble Communicating With Gateway&#34; message.  Googling the message returns one web page in German.  Retried three times with the same result.  So...  Looks like I'm screwed. 

I bought this crapware online but was not able to install it on my tablet nor my android phone. The amazon android market app \\"seems\\" to crash when installing free or promotional apps. Amazon customer service is non existent. They 

**How many reviews had line breaks before cleaning, and after?**

25,699 reviews had `<br />` line breaks before cleaning and 0 after.

**What still looks weird in the five reviews? Do you think it would change whether a model reads a review as positive or negative?**

The `&#34;` codes and the `\\"` quotes are still there. I don't think this would change whether a model reads a review as positive or negative because the sentiment comes from the actual words like "Perfect" or "crapware," not from the quote marks.

### 8. Take a sample

In [7]:
sample = df.groupby("star_rating").sample(n=100, random_state=42)
complaints = df[df["star_rating"] == 1].sample(n=20, random_state=42)
print(len(sample), len(complaints))

500 20


**Why take the same number of each star rating instead of 500 random reviews?**

Since over half of the reviews are 5 stars a random sample of 500 would be mostly 5-star reviews and only have around 25 two-star reviews. Taking 100 of each rating makes sure the model gets tested the same amount on every rating, so I can see how it does on each one instead of just the most common one.

## Part 2: Run a Sentiment Model
### 9. Try one review

In [8]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
)
print(sample["star_rating"].iloc[0], sample["review_body"].iloc[0][:200])
classifier(sample["review_body"].iloc[0])

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

1 Terrible, confusing interface. Looks like a cartoon. Didn't really like the long, drawn out stories in my player. Game play and graphics weren't that much improved $30 extra we had to pay (Now we get 


[{'label': 'NEGATIVE', 'score': 0.9998071789741516}]

**What are the keys in the result, and what does the score mean?**

The keys are `label` and `score`. The label is what the model thinks the sentiment is (POSITIVE or NEGATIVE), and the score is how confident the model is in that label, from 0 to 1.

**Does the model agree with the customer's star rating?**

Yes. The review was 1 star and the model said NEGATIVE with a score of 0.9998, so it agrees and is very confident.

### 10. Classify all 500 reviews in one call

In [9]:
results = classifier(sample["review_body"].tolist(), truncation=True)
sample["label"] = [r["label"] for r in results]
sample["score"] = [r["score"] for r in results]
sample[["star_rating", "label", "score"]].head(10)

,star_rating,label,score
40836,1,NEGATIVE,0.999807
85497,1,NEGATIVE,0.990978
124434,1,NEGATIVE,0.998919
118528,1,NEGATIVE,0.999704
13480,1,NEGATIVE,0.999466
47179,1,NEGATIVE,0.999234
130193,1,NEGATIVE,0.999613
126902,1,NEGATIVE,0.999815
58985,1,NEGATIVE,0.999799
22608,1,NEGATIVE,0.999736


**About how long did 500 reviews take? At that rate, how long would all 145,000 take?**

The 500 reviews took about 35 seconds. At that rate, all 145,000 reviews would take around 10,150 seconds, which is almost 3 hours.

**What does `truncation=True` protect against?**

The model can only read a limited number of tokens at a time so a really long review would cause an error. `truncation=True` cuts off anything past that limit, so long reviews don't crash the whole call.

## Part 3: Is the Model Any Good?
### 11. Create the answer key and score the model

In [10]:
def stars_to_sentiment(stars):
    """Convert a star rating into the sentiment the customer expressed."""
    if stars <= 2:
        return "NEGATIVE"
    if stars >= 4:
        return "POSITIVE"
    return "NEUTRAL"


sample["truth"] = sample["star_rating"].apply(stars_to_sentiment)
graded = sample[sample["truth"] != "NEUTRAL"]
print("Accuracy:", (graded["label"] == graded["truth"]).mean())

Accuracy: 0.8775


**Why do we leave out the 3-star reviews here?**

A 3-star review is neutral or mixed, so there isn't a correct answer when the model can only pick POSITIVE or NEGATIVE. Including them would count the model as wrong for reviews that don't really have a right answer.

**What's the accuracy? Would you trust this model in a product?**

The accuracy is 0.8775, so about 88%. I would trust it for something like showing overall trends, since the mistakes would average out, but not for decisions about single reviews, because it still gets about 1 in 8 wrong.

### 12. Break it down by star rating

In [11]:
pd.crosstab(sample["star_rating"], sample["label"])

label,NEGATIVE,POSITIVE
star_rating,,
1,96,4
2,95,5
3,66,34
4,23,77
5,17,83


**Which star rating does the model get wrong most often?**

The model gets 4-star reviews wrong the most, labeling 23 of the 100 as NEGATIVE. It also got 17 of the 5-star reviews wrong.

**The model only knows POSITIVE and NEGATIVE. What did it do with the 3-star reviews?**

It was forced to pick one, so it labeled 66 of them NEGATIVE and 34 POSITIVE. Therefore, it leans negative on mixed reviews.

### 13. Read the mistakes

In [12]:
mistakes = graded[graded["label"] != graded["truth"]]
for review in mistakes.head(5).itertuples():
    print(review.star_rating, review.label, round(review.score, 2))
    print(review.review_body[:300], "\n")

1 POSITIVE 0.99
Don't get me wrong, the game concept is awesome. Graphics are great.  BUT  If you don't want to lose your progress when the Simcity Servers stop responding, don't buy this. There is no option to save locally just saved to their server so the smallest error will lose your work.  Overall it's a good g 

1 POSITIVE 1.0
Enjoys game 

1 POSITIVE 0.73
I wasted $7. im sure its a good game but i cant even get to the game part. i tried. a lot. 

1 POSITIVE 1.0
Dose not work right on PSVITA so was not happy but it is a fun game love twisted metal games. 

2 POSITIVE 0.87
I bought this game for my 8 year old daughter who likes the kingdom of keflings game on xbox. The graphics are very simple and it is a basic strategy/building style game. Unfortunately, it looks like it would have been cutting edge 15 to 20 years ago, but it really didn't entertain my kids for more  



**For each of the five, why do you think the model got it wrong? Was it the model, or was the customer's rating misleading?**

1. SimCity (1 star, POSITIVE): The review starts with "the game concept is awesome. Graphics are great" before the "BUT" about losing progress. The model most likely got thrown off by the positive words at the start, so this was the model's fault.
2. "Enjoys game" (1 star, POSITIVE): The text is completely positive, so the customer's rating was misleading here. They most likely clicked the wrong star or were rating something else.
3. "I wasted $7" (1 star, POSITIVE, 0.73): The customer says "im sure its a good game" but they couldn't play it. The model picked up on "good game" so this was the model, although its lower score shows it wasn't sure.
4. PS Vita (1 star, POSITIVE): The customer says it doesn't work right but also that "it is a fun game love twisted metal games." The rating is about the device problem while most of the words are positive, so I think the rating was more misleading than the model.
5. Keflings (2 stars, POSITIVE): The complaint is subtle, saying it looks like it would have been cutting edge 15 to 20 years ago and didn't entertain the kids. The model missed that, so this one was the model's fault.

## Part 4: A Model That Predicts Stars
### 14. Predict the stars

In [13]:
star_model = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
)
results = star_model(sample["review_body"].tolist(), truncation=True)
sample["pred_stars"] = [int(r["label"][0]) for r in results]  # "4 stars" -> 4

print("Exact:", (sample["pred_stars"] == sample["star_rating"]).mean())
off_by = (sample["pred_stars"] - sample["star_rating"]).abs()
print("Within one star:", (off_by <= 1).mean())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Exact: 0.574
Within one star: 0.912


**How often does it get the exact star rating? How often within one star?**

It gets the exact star rating 57.4% of the time and is within one star 91.2% of the time.

**Your team is building a sentiment dashboard for a game publisher. Which of the two models would you use, and why?**

I would use the star model. It gives a full 1 to 5 rating, so it can handle mixed 3-star reviews instead of forcing them into positive or negative, and it was within one star 91% of the time. The first model only gives two labels , so a publisher would lose a lot of detail in their dashboard.

## Part 5: Zero-Shot - What Are People Complaining About?
### 15. Classify 20 one-star reviews

In [14]:
zsc = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

labels = ["price", "bugs or crashes", "account or login", "gameplay", "download or install"]
results = zsc(complaints["review_body"].tolist(), candidate_labels=labels)
complaints["topic"] = [r["labels"][0] for r in results]
complaints["topic"].value_counts()

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

topic
download or install    5
bugs or crashes        5
account or login       4
price                  3
gameplay               3
Name: count, dtype: int64

**What do customers complain about most?**

"Download or install" and "bugs or crashes" are tied for the most with 5 reviews each, followed by "account or login" with 4. Therefore, half of the 20 complaints are about technical problems getting the game to work, not the game itself.

In [15]:
for review in complaints.head(3).itertuples():
    print(review.topic, "|", review.review_body[:300], "\n")

price | Terrible, confusing interface. Looks like a cartoon. Didn't really like the long, drawn out stories in my player. Game play and graphics weren't that much improved $30 extra we had to pay (Now we get VC yea! not). 

download or install | I too installed and tried to play Cobi Treasure Island. My PC does meet the proper requirements so I do not know what the problem is. 

gameplay | If you liked \\"Heavy Rain\\", you'll like this.  However, if you'd like for your games to have a faster pace than 'the English Patient', you'll want to skip this title in favor of something more exciting, like \\"Parking Simulator 4\\". 



**Does the model's choice make sense?**

Mostly. the second review (download or install) is about a game that won't run after installing, and the third (gameplay) complains about the slow pace, so both make sense. The first one was labeled price because it mentions paying $30 extra, but most of the review is about the confusing interface and story, so "gameplay" would have been a better fit.

### 16. Change the labels

In [16]:
more_labels = labels + ["technical problems", "HR policy"]
new_results = zsc(complaints["review_body"].head(5).tolist(), candidate_labels=more_labels)
for old, new in zip(results[:5], new_results):
    print("Before:", old["labels"][0], round(old["scores"][0], 2))
    print("After: ", new["labels"][0], round(new["scores"][0], 2), "\n")

Before: price 0.66
After:  price 0.59 

Before: download or install 0.41
After:  download or install 0.3 

Before: gameplay 0.36
After:  gameplay 0.32 

Before: bugs or crashes 0.53
After:  technical problems 0.7 

Before: download or install 0.79
After:  download or install 0.45 



**What happened to the top labels and their scores?**

Four of the five reviews kept the same top label but every score went down. For one review the top label switched from "bugs or crashes" (0.53) to the new "technical problems" label (0.70).

**The scores for each review add up to 1. How does that explain what you saw?**

Since the scores always add up to 1, adding more labels means the same total has to be split more. The overlapping label "technical problems" took a big share from labels like "bugs or crashes" and "download or install," which is why those scores dropped and one top label changed. The irrelevant "HR policy" label only took a small share, but it still took some.

**What does this tell you about choosing labels for a real product?**

The labels need to be clearly different from each other because the scores are only relative to the labels you give. Overlapping labels split the score between them, and changing the labels can change the results, so a real product should test its labels and keep them the same over time so the results can be compared.